In [120]:
import numpy as np
import pandas as pd
from typing import Optional
from matplotlib import pyplot as plt
import matplotlib.ticker as plticker
import seaborn as sns
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [121]:
outcome_var = "log_measurement"
intervention_year: Optional[int] = None
intervention_year: Optional[int] = 2015

treatment_var = "Regions"
# treatment_var_value = "Gulf of Mexico"
treatment_var_value = "Mediterranean Sea"
treatment_var_value = "North Sea"
treatment_var_value = "Gulf of California"
treatment_var_value = "Caribbean Sea"

control_var_dummy_value = "Other Seas"


filepath = "/mnt/d/temp/user/ed/mart/mmplastic/Marine_Microplastics_WGS84_1130715472761438990.csv"

In [122]:
df = pd.read_csv(filepath)
df = df.dropna(subset=["Measurement"])
df['yyyymmdd'] = pd.to_datetime(df['Date'])
df['year'] = df['yyyymmdd'].dt.year
df['month'] = df['yyyymmdd'].dt.month
df['day'] = df['yyyymmdd'].dt.day

df = df.rename(columns={"Sampling Method": "sampling_method", "Density Class": "density"})

df["measurement"] = np.where(df["Measurement"] > 1.0, 1.0, df["Measurement"])
df["log_measurement"] = np.log(df["measurement"])

/tmp/ipykernel_263785/4292777884.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['yyyymmdd'] = pd.to_datetime(df['Date'])
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/pandas/core/arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Min and Max Year of Treatment Var.

In [123]:
year_range = df[df[treatment_var] == treatment_var_value].agg({"year": ["min", "max"]})
if intervention_year is None:
    intervention_year = int((year_range.loc["min"] + (year_range.loc["max"] - year_range.loc["min"])/2).loc["year"])
(
    year_range,
    intervention_year
)

(     year
 min  1986
 max  2021,
 2015)

## Find lower/higher quantile thresholds

In [124]:
def find_quantile_below_threshold(df, column, threshold, quantile_range=(0, 1, 0.005)):
  """
  Finds the highest quantile value less than a given threshold in a Pandas DataFrame column.

  Args:
    df: The Pandas DataFrame.
    column: The name of the column to check.
    threshold: The threshold value.
    quantile_range: A tuple defining the start, end, and step of the quantile range.

  Returns:
    The quantile value that is less than the threshold.
  """

  quantiles = np.arange(*quantile_range)
  for q in quantiles[::-1]:
    quantile_value = df[column].quantile(q)
    if quantile_value < threshold:
      return q, quantile_value
  return None, None  # If no quantile is found

threshold = 0.0
quantile, value = find_quantile_below_threshold(df, 'log_measurement', threshold)
print("Quantile:", quantile)
print("Value:", value)

higher_quantile = quantile

Quantile: 0.86
Value: -0.024470811007744744


In [125]:
def find_quantile_over_threshold(df, column, threshold, quantile_range=(0, 1, 0.005)):
  """
  Finds the lowest quantile value less than a given threshold in a Pandas DataFrame column.

  Args:
    df: The Pandas DataFrame.
    column: The name of the column to check.
    threshold: The threshold value.
    quantile_range: A tuple defining the start, end, and step of the quantile range.

  Returns:
    The quantile value that is less than the threshold.
  """

  quantiles = np.arange(*quantile_range)
  for q in quantiles:
    quantile_value = df[column].quantile(q)
    if quantile_value > threshold:
      return q, quantile_value
  return None, None  # If no quantile is found

threshold = -10.0
quantile, value = find_quantile_over_threshold(df, outcome_var, threshold)
print("Quantile:", quantile)
print("Value:", value)

lower_quantile = quantile

Quantile: 0.32
Value: -6.907755278982137


/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: inva

In [126]:
higher_threshold = np.ceil(df[outcome_var].quantile(higher_quantile) * 100.) / 100.
lower_threshold = np.ceil(df[outcome_var].quantile(lower_quantile) * 1000.) / 1000.

In [127]:
df = df.loc[
    (df[outcome_var] > lower_threshold) & (df[outcome_var] < higher_threshold),
    :
]

## Set post variable
- potential intervention year

In [128]:
df["post"] = (df["year"] > intervention_year).apply(lambda a: int(a))

## Set treated
- intervention region

In [129]:
treatment_condition = (df[treatment_var] == treatment_var_value)
control_condition = (df[treatment_var] != treatment_var_value)

In [130]:
df["treated"] = (treatment_condition).apply(lambda a: int(a))
# df["treated"] = (df["Oceans"] == "Pacific Ocean").apply(lambda a: int(a))

## Fix dataset

In [131]:
df = df.loc[:, [outcome_var, "year", "treated", "post", "density", treatment_var, "sampling_method"]]

In [132]:
uniq_df = df.dropna(subset=[treatment_var]).groupby([treatment_var, "year"]).agg({outcome_var: "mean"}).drop_duplicates() #.reset_index()

In [133]:
uniq_df

log_measurement
Regions                                   year                 
Baffin Bay                                2016        -5.518964
                                          2021        -3.918329
Baltic Sea                                2018        -3.984331
                                          2019        -2.441691
                                          2020        -4.405978
...                                                         ...
Stellwagen Bank National Marine Sanctuary 2001        -3.365174
                                          2002        -6.157721
                                          2003        -4.067257
                                          2004        -3.744791
                                          2006        -5.134160

[160 rows x 1 columns]

In [134]:
df5 = df.set_index([treatment_var, "year"]).join(
    uniq_df, on=[treatment_var, "year"], rsuffix="_mean", how="left"
).drop(columns=outcome_var).rename(columns={f"{outcome_var}_mean": outcome_var}).loc[:, ["treated", "post", outcome_var]].drop_duplicates().reset_index().dropna(subset=[treatment_var]).astype({"year": "str"})
# check duplicates
# df5[df5.duplicated(subset=[treatment_var, "year"], keep=False)]
df = df5

In [135]:
df

,Regions,year,treated,post,log_measurement
1,North Sea,2019,0,1,-4.118103
3,Mediterranean Sea,2018,0,1,-2.106566
4,Caribbean Sea,1999,1,0,-4.811461
5,Caribbean Sea,2003,1,0,-4.908958
6,Caribbean Sea,1993,1,0,-5.277655
...,...,...,...,...,...
157,Barents Sea,2016,0,1,-4.509860
158,Celtic Sea,2017,0,1,-3.995242
159,Bay of Biscay,2015,0,0,-2.255902
160,Caribbean Sea,2018,1,1,-5.222468


# Matrix Representation

In [136]:
treated = list(df.query("treated == 1")[treatment_var].unique())
treated

['Caribbean Sea']

In [137]:
tr_period = df.query("post == 1")["year"].min()
tr_period

'2016'

In [138]:
def reshape_sc_data(df: pd.DataFrame,
                    geo_col: str, 
                    time_col: str,
                    y_col: str,
                    tr_geos: str,
                    tr_start: str):
    
    df_pivot = df.pivot(index=time_col, columns=geo_col, values=y_col)
    
    y_co = df_pivot.drop(columns=tr_geos)
    y_tr = df_pivot[tr_geos]
    
    y_pre_co = y_co[df_pivot.index < tr_start]
    y_pre_tr = y_tr[df_pivot.index < tr_start]
    
    y_post_co = y_co[df_pivot.index >= tr_start]
    y_post_tr = y_tr[df_pivot.index >= tr_start]
    
    return y_pre_co, y_pre_tr, y_post_co, y_post_tr

In [139]:
y_pre_co, y_pre_tr, y_post_co, y_post_tr = reshape_sc_data(
    df,
    geo_col=treatment_var,
    time_col="year",
    y_col=outcome_var,
    tr_geos=treated,
    tr_start=str(tr_period)
)

y_pre_tr.head()

Regions,Caribbean Sea
year,
1987,-5.056555
1989,-5.365995
1990,-5.444500
1991,-5.467754
1992,-5.092289


In [140]:
y_pre_tr

Regions,Caribbean Sea
year,
1987,-5.056555
1989,-5.365995
1990,-5.444500
1991,-5.467754
1992,-5.092289
1993,-5.277655
1994,-5.234888
1995,-4.921982
1996,-4.671041
